# Modeling Human Activity States with Hidden Markov Models

**Use case:** elderly fall-risk and inactivity monitoring.

A Hidden Markov Model, implemented from scratch in NumPy, infers four hidden activity states (**still, standing, walking, jumping**) from smartphone accelerometer and gyroscope data.

**Sections**
1. Setup & data loading
2. Pre-processing & sampling-rate harmonisation
3. Feature extraction
4. HMM implementation
5. Model verification
6. Training
7. Evaluation on unseen data

## 1. Setup & Data Loading

Recordings were made with **Sensor Logger** (Android, Infinix X6855). Each export is a folder with one CSV per sensor. Each CSV has `time` (ns since the Unix epoch), `seconds_elapsed`, and the axes in the order `z, y, x`. A `Metadata.csv` file records the device and the configured sample rates.

Two streams are used:

| Stream | File | Units | Why |
|---|---|---|---|
| Accelerometer | `TotalAcceleration.csv` | m/s² | The true accelerometer reading, **including gravity**. The app's `Accelerometer.csv` has gravity removed, which would discard the orientation cue that separates *still* (phone flat) from *standing* (phone upright). |
| Gyroscope | `Gyroscope.csv` | rad/s | Calibrated angular velocity (bias-corrected). |

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
LABELS_DIR = ROOT / "data" / "labels"
FIG_DIR = ROOT / "figures"

ACTIVITIES = ["still", "standing", "walking", "jumping"]
SENSOR_FILES = {"acc": "TotalAcceleration.csv", "gyro": "Gyroscope.csv"}
TARGET_FS = 25  # Hz; common rate every recording is harmonised to (Section 2)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.precision", 3)

### 1.1 Loading utilities

In [2]:
def load_metadata(rec_dir):
    """Read a recording's Metadata.csv.

    Returns a dict with the device, OS and app versions, the recording start
    (epoch, ns) and the configured sample rate in Hz for each stream in
    SENSOR_FILES.
    """
    meta = pd.read_csv(rec_dir / "Metadata.csv").iloc[0]
    rate_ms = dict(zip(meta["sensors"].split("|"), str(meta["sampleRateMs"]).split("|")))
    configured_hz = {
        key: 1000 / float(rate_ms[fname.removesuffix(".csv")])
        for key, fname in SENSOR_FILES.items()
    }
    return {
        "device": meta["device name"],
        "platform": f'{meta["platform"]} {meta["platform version"]}',
        "app_version": str(meta["appVersion"]),
        "recorded_at": meta["recording time"],
        "epoch_ns": int(meta["recording epoch time"]) * 1_000_000,
        "configured_hz": configured_hz,
    }


def load_sensor(rec_dir, sensor, epoch_ns):
    """Load one sensor stream of a recording.

    Parameters
    ----------
    rec_dir : Path
        Folder of one exported recording.
    sensor : str
        Key of SENSOR_FILES ("acc" or "gyro").
    epoch_ns : int
        Recording start (ns), so every stream of a recording shares one time origin.

    Returns
    -------
    DataFrame with columns timestamp_ns, t (s since recording start), x, y, z,
    sorted by time with duplicate timestamps removed.
    """
    df = pd.read_csv(rec_dir / SENSOR_FILES[sensor])
    df = df.sort_values("time").drop_duplicates("time")
    return pd.DataFrame({
        "timestamp_ns": df["time"].to_numpy(),
        "t": (df["time"].to_numpy() - epoch_ns) / 1e9,
        "x": df["x"].to_numpy(),
        "y": df["y"].to_numpy(),
        "z": df["z"].to_numpy(),
    })


def load_recording(rec_dir):
    """Load metadata plus the accelerometer and gyroscope streams of one recording."""
    rec_dir = Path(rec_dir)
    meta = load_metadata(rec_dir)
    rec = {"name": rec_dir.name, "meta": meta}
    for sensor in SENSOR_FILES:
        rec[sensor] = load_sensor(rec_dir, sensor, meta["epoch_ns"])
    return rec


def sampling_stats(df):
    """Measured sampling statistics of one stream.

    The measured rate is 1 / median(dt). "Dropped" counts intervals longer than
    1.5x the median (at least one missing sample). "Large gaps" counts intervals
    longer than 3x the median, which would need attention during cleaning.
    """
    dt = np.diff(df["t"].to_numpy())
    med = np.median(dt)
    return {
        "n_samples": len(df),
        "duration_s": df["t"].iloc[-1] - df["t"].iloc[0],
        "measured_hz": 1 / med,
        "median_dt_ms": med * 1e3,
        "jitter_ms": dt.std() * 1e3,
        "max_dt_ms": dt.max() * 1e3,
        "dropped": int((dt > 1.5 * med).sum()),
        "large_gaps": int((dt > 3 * med).sum()),
    }

### 1.2 Pilot check

Before the main collection, four short pilot recordings were made to check the data format:

| Pilot | Phone | Question it answers |
|---|---|---|
| `pilot_01_flat` | Flat on a table | Is gravity included? Which axis carries it when flat? |
| `pilot_02_upright` | Held at waist level (the grip used for standing, walking, jumping) | Which axis carries gravity when upright? |
| `pilot_03_walk` | Same grip, walking | Data quality during movement |
| `pilot_04_flat_50hz` | Flat, app set to a lower rate | Does the rate setting change the delivered rate? |

In [3]:
pilots = [load_recording(p) for p in sorted((RAW_DIR / "pilot").iterdir()) if p.is_dir()]

print(f"Device: {pilots[0]['meta']['device']}  |  {pilots[0]['meta']['platform']}  |  Sensor Logger {pilots[0]['meta']['app_version']}")
pilot_rates = pd.DataFrame([
    {"recording": rec["name"], "sensor": sensor,
     "configured_hz": rec["meta"]["configured_hz"][sensor], **sampling_stats(rec[sensor])}
    for rec in pilots for sensor in SENSOR_FILES
]).set_index(["recording", "sensor"])
pilot_rates

Device: Infinix X6855  |  android 36  |  Sensor Logger 1.68


configured_hz  n_samples  duration_s  measured_hz  \
recording          sensor                                                      
pilot_01_flat      acc             100.0        574       9.804       58.445   
                   gyro            100.0        575       9.825       58.445   
pilot_02_upright   acc             100.0        736      12.575       58.448   
                   gyro            100.0        739      12.635       58.448   
pilot_03_walk      acc             100.0        898      15.342       58.452   
                   gyro            100.0        897      15.329       58.452   
pilot_04_flat_50hz acc              50.0        528      18.031       29.227   
                   gyro             50.0        529      18.066       29.227   

                           median_dt_ms  jitter_ms  max_dt_ms  dropped  \
recording          sensor                                                
pilot_01_flat      acc           17.110  1.517e-04     17.110        0   
                   gyro          17.110  1.784e-01     21.387        0   
pilot_02_upright   acc           17.109  1.330e-04     17.110        0   
                   gyro          17.109  3.147e-01     25.664        0   
pilot_03_walk      acc           17.108  1.427e-01     17.109        0   
                   gyro          17.108  1.627e-04     17.109        0   
pilot_04_flat_50hz acc           34.215  1.718e-04     34.216        0   
                   gyro          34.215  1.809e-04     34.216        0   

                           large_gaps  
recording          sensor              
pilot_01_flat      acc              0  
                   gyro             0  
pilot_02_upright   acc              0  
                   gyro             0  
pilot_03_walk      acc              0  
                   gyro             0  
pilot_04_flat_50hz acc              0  
                   gyro             0

In [4]:
def stream_offset_ms(rec):
    """Start-time difference between the gyroscope and accelerometer streams (ms)."""
    return (rec["gyro"]["t"].iloc[0] - rec["acc"]["t"].iloc[0]) * 1e3


def shared_timestamps(rec):
    """Fraction of accelerometer timestamps that also appear in the gyroscope stream."""
    return np.isin(rec["acc"]["timestamp_ns"], rec["gyro"]["timestamp_ns"]).mean()


pd.DataFrame([
    {"recording": rec["name"], "gyro_start_offset_ms": stream_offset_ms(rec),
     "shared_timestamps": shared_timestamps(rec)}
    for rec in pilots
]).set_index("recording")

,gyro_start_offset_ms,shared_timestamps
recording,,
pilot_01_flat,-17.110,0.0
pilot_02_upright,-17.109,0.0
pilot_03_walk,17.108,0.0
pilot_04_flat_50hz,-17.108,0.0


In [5]:
def steady(df, trim_s=1.0):
    """Drop the first and last trim_s seconds (button-press and settling artefacts)."""
    t = df["t"]
    return df[(t > t.iloc[0] + trim_s) & (t < t.iloc[-1] - trim_s)]


def magnitude(df):
    """Euclidean norm of the x, y, z columns."""
    return np.sqrt(df["x"] ** 2 + df["y"] ** 2 + df["z"] ** 2)


orientation = pd.DataFrame([
    {"recording": rec["name"],
     **{f"mean_acc_{ax}": steady(rec["acc"])[ax].mean() for ax in "xyz"},
     "mean_|a|": magnitude(steady(rec["acc"])).mean(),
     "std_|a|": magnitude(steady(rec["acc"])).std(),
     "std_|w|": magnitude(steady(rec["gyro"])).std()}
    for rec in pilots
]).set_index("recording")
orientation

,mean_acc_x,mean_acc_y,mean_acc_z,mean_|a|,std_|a|,std_|w|
recording,,,,,,
pilot_01_flat,-0.081,0.578,9.819,9.836,0.007,4.311e-04
pilot_02_upright,0.369,9.843,1.286,9.944,0.142,1.044e-01
pilot_03_walk,0.299,9.857,1.094,9.992,0.852,4.798e-01
pilot_04_flat_50hz,-0.137,0.502,9.826,9.840,0.048,6.309e-03


### 1.3 Pilot findings

- **Gravity.** `TotalAcceleration` has |a| ≈ 9.8 m/s² at rest, so it includes gravity and is used as the accelerometer.
- **Orientation separates still from standing.** Flat on the table, gravity lies on **z** (≈ 9.8 m/s²). In the waist-level grip it lies on **y** (≈ 9.7 m/s², z ≈ 1.3). The per-axis means of the accelerometer are therefore strong features for still vs standing.
- **Delivered rate ≠ configured rate.** A configured 100 Hz delivers **≈ 58.4 Hz** (17.1 ms). The lower setting (50 Hz) delivers **≈ 29.2 Hz** (34.2 ms). The phone snaps requests to its own supported rates.
- **Timing quality.** Sampling is very regular: jitter σ ≤ 0.3 ms, no dropped samples and no gaps. The gyroscope has a few slightly long intervals (up to ~26 ms).
- **Streams are not aligned.** The accelerometer and gyroscope are sampled at different instants (**no shared timestamps**) and start up to one sample period (~17 ms) apart, so the two streams cannot be joined row by row. Both are interpolated onto one common time grid instead.

**Consequence for harmonisation (Section 2).** Recordings exist at two real rates (58.4 Hz and 29.2 Hz). All streams are low-pass filtered (anti-aliasing) and resampled onto a common **25 Hz** grid. 25 Hz sits just below the lowest delivered rate, so every recording is *downsampled*; nothing is upsampled. The resulting Nyquist frequency of 12.5 Hz still covers the walking and jumping fundamentals (~1.5–2.5 Hz) and their main harmonics.